Here we will add our raw data of the chosen domain.

In [13]:
import csv
with open ("\\HCL TRAINING\\DATA\\RAW\\electricity_data.csv", newline="", encoding='utf-8') as f:
    reader = csv.reader(f)
    header = next(reader)
    print('columns:',header)

    for i, row in enumerate(reader):
        if i<255:
            print(row) 

columns: ['Household_ID', 'City', 'Property_Type', 'Tariff_Plan', 'Has_Solar', 'Household_Size', 'Home_Area_Sqft', 'AC_Hours_Per_Day', 'Avg_Temperature_C', 'Appliance_Age_Years', 'Reading_Date', 'Monthly_Consumption_kWh']
['H1091', 'Mumbai', 'Apartment', 'Standard', 'No', '2', '900', '1.5', '30.7', '9', '2026-10-07', '166']
['H1133', 'Chennai', 'Apartment', 'Standard', 'No', '2', '1550', '7.9', '24.0', '5', '2026-02-06', '324']
['H1084', 'Mumbai', 'Independent House', 'Time of Day', 'No', '5', '3020', '7.1', '26.0', '5', '2026-03-08', '402']
['H1056', 'Bengaluru', 'Apartment', 'Prepaid Smart', 'No', '4', '740', '5.3', '19.3', '5', '2025-02-02', '175']
['H1129', 'Bengaluru', 'Apartment', 'Standard', 'Yes', '6', '460', '7.4', '25.2', '8', '2025-07-11', '217']
['H1061', 'Delhi', 'Apartment', 'Standard', 'No', '2', '990', '5.5', '27.0', '4', '2025-03-05', '262']
['H1102', 'Delhi', 'Apartment', 'Standard', 'Yes', '1', '1190', '6.9', '16.1', '6', '2026-01-22', '167']
['H1070', 'Kolkata', 'In

Cleaning the raw data into a new csv file with only the required columns and rows.


In [14]:
import os
import pandas as pd
import numpy as np

# Run this from the "notebooks" folder so the relative paths below work:
# household-electricity-consumption/
#   data/raw/electricity_data.csv        <- input
#   data/processed/                      <- output
#   notebooks/                           <- this script / notebook

RAW_PATH = "../data/raw/electricity_data.csv"
CLEAN_PATH = "../data/processed/electricity_data_clean.csv"

# ------------------------------------------------------------------
# 1. LOAD THE RAW DATA
# ------------------------------------------------------------------
df = pd.read_csv(RAW_PATH)
print("Raw shape:", df.shape)


# ------------------------------------------------------------------
# 2. FIRST LOOK (inspect before cleaning)
# ------------------------------------------------------------------
print(df.info())
print("\nMissing values per column:\n", df.isna().sum())
print("\nDuplicate rows:", df.duplicated().sum())
print("\nNumeric summary:\n", df.describe())


# ------------------------------------------------------------------
# 3. REMOVE DUPLICATE ROWS
# ------------------------------------------------------------------
df = df.drop_duplicates().reset_index(drop=True)
print("\nAfter removing duplicates:", df.shape)


# ------------------------------------------------------------------
# 4. FIX INCONSISTENT TEXT (extra spaces, upper/lower case)
# ------------------------------------------------------------------
cat_cols = ["City", "Property_Type", "Tariff_Plan", "Has_Solar"]
for col in cat_cols:
    df[col] = df[col].str.strip().str.title()
df["Tariff_Plan"] = df["Tariff_Plan"].replace({"Time Of Day": "Time of Day"})

for col in cat_cols:
    print(f"\n{col} values:\n", df[col].value_counts(dropna=False))


# ------------------------------------------------------------------
# 5. HANDLE IMPOSSIBLE VALUES
#    Valid ranges come from the data dictionary.
#    Invalid values are turned into NaN so they are treated as missing.
# ------------------------------------------------------------------
valid_ranges = {
    "Household_Size": (1, 7),
    "Home_Area_Sqft": (350, 3200),
    "AC_Hours_Per_Day": (0, 24),
    "Avg_Temperature_C": (5, 46),
    "Appliance_Age_Years": (0, 20),
    "Monthly_Consumption_kWh": (40, 1500),
}
for col, (low, high) in valid_ranges.items():
    bad = (df[col] < low) | (df[col] > high)
    print(f"{col}: {bad.sum()} impossible value(s)")
    df.loc[bad, col] = np.nan


# ------------------------------------------------------------------
# 6. FIX DATA TYPES
# ------------------------------------------------------------------
df["Reading_Date"] = pd.to_datetime(df["Reading_Date"], errors="coerce")


# ------------------------------------------------------------------
# 7. HANDLE MISSING VALUES
# ------------------------------------------------------------------
# Categorical column: fill with the most common value (mode)
df["Tariff_Plan"] = df["Tariff_Plan"].fillna(df["Tariff_Plan"].mode()[0])

# Home area depends on property type, so use the median of each type
df["Home_Area_Sqft"] = df["Home_Area_Sqft"].fillna(
    df.groupby("Property_Type")["Home_Area_Sqft"].transform("median"))

# Other numeric columns: fill with the overall median
# (Household_Size and Monthly_Consumption_kWh are included so that any value
#  turned into NaN in step 5 is also handled.)
for col in ["AC_Hours_Per_Day", "Avg_Temperature_C", "Appliance_Age_Years",
            "Household_Size", "Monthly_Consumption_kWh"]:
    df[col] = df[col].fillna(df[col].median())

# Round columns that should be whole numbers, then convert to int
for col in ["Home_Area_Sqft", "Appliance_Age_Years",
            "Household_Size", "Monthly_Consumption_kWh"]:
    df[col] = df[col].round().astype(int)
df["AC_Hours_Per_Day"] = df["AC_Hours_Per_Day"].round(1)
df["Avg_Temperature_C"] = df["Avg_Temperature_C"].round(1)


# ------------------------------------------------------------------
# 8. FINAL CHECKS (the script stops with an error if something is wrong)
# ------------------------------------------------------------------
print("\nClean shape:", df.shape)
print("Missing values left:", df.isna().sum().sum())
print("Duplicate rows left:", df.duplicated().sum())
print("Duplicate IDs left:", df["Household_ID"].duplicated().sum())

assert df.isna().sum().sum() == 0, "Missing values are still present"
assert df.duplicated().sum() == 0, "Duplicate rows are still present"
assert df["Household_ID"].is_unique, "Household_ID is not unique"
assert df["City"].nunique() == 5, "City spelling is still inconsistent"
assert df["Tariff_Plan"].nunique() == 3, "Tariff_Plan has unexpected values"
for col, (low, high) in valid_ranges.items():
    assert df[col].between(low, high).all(), f"{col} still has impossible values"
print("All final checks passed.")
print(df.describe())


# ------------------------------------------------------------------
# 9. SAVE THE CLEAN DATA
# ------------------------------------------------------------------
os.makedirs(os.path.dirname(CLEAN_PATH), exist_ok=True)
df.to_csv(CLEAN_PATH, index=False)
print("\nSaved:", CLEAN_PATH)


Raw shape: (255, 12)
<class 'pandas.DataFrame'>
RangeIndex: 255 entries, 0 to 254
Data columns (total 12 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Household_ID             255 non-null    str    
 1   City                     255 non-null    str    
 2   Property_Type            255 non-null    str    
 3   Tariff_Plan              251 non-null    str    
 4   Has_Solar                255 non-null    str    
 5   Household_Size           255 non-null    int64  
 6   Home_Area_Sqft           250 non-null    float64
 7   AC_Hours_Per_Day         255 non-null    float64
 8   Avg_Temperature_C        250 non-null    float64
 9   Appliance_Age_Years      249 non-null    float64
 10  Reading_Date             255 non-null    str    
 11  Monthly_Consumption_kWh  255 non-null    int64  
dtypes: float64(4), int64(2), str(6)
memory usage: 24.0 KB
None

Missing values per column:
 Household_ID               0
City   

Here we will add new columns.


In [15]:
import pandas as pd
import numpy as np

# Run from the "notebooks" folder (same as the cleaning script).
CLEAN_PATH = "../data/processed/electricity_data_clean.csv"

# ------------------------------------------------------------------
# 1. LOAD THE CLEANED DATA
# ------------------------------------------------------------------
df = pd.read_csv(CLEAN_PATH, parse_dates=["Reading_Date"])
print("Loaded:", df.shape)


# ------------------------------------------------------------------
# 2. NEW COLUMN 1: Usage_Per_Sqft  (combines two existing columns)
#    electricity used for every square foot of the home
# ------------------------------------------------------------------
df["Usage_Per_Sqft"] = (df["Monthly_Consumption_kWh"] / df["Home_Area_Sqft"]).round(3)


# ------------------------------------------------------------------
# 3. NEW COLUMN 2: Usage_Status  (label made with a condition)
#    High Usage if the household uses more than 350 kWh in the month
# ------------------------------------------------------------------
df["Usage_Status"] = np.where(df["Monthly_Consumption_kWh"] > 350, "High Usage", "Normal")


# ------------------------------------------------------------------
# 4. NEW COLUMN 3: AC_Usage_Band  (numeric column grouped into bins)
#    Low: up to 3 hours, Medium: more than 3 up to 7, Heavy: more than 7
# ------------------------------------------------------------------
df["AC_Usage_Band"] = pd.cut(
    df["AC_Hours_Per_Day"],
    bins=[-0.1, 3, 7, 24],
    labels=["Low (0-3)", "Medium (3-7)", "Heavy (7+)"],
)


# ------------------------------------------------------------------
# 5. CHECK THE NEW COLUMNS
# ------------------------------------------------------------------
print(df[["Household_ID", "Monthly_Consumption_kWh", "Home_Area_Sqft", "Usage_Per_Sqft",
          "Usage_Status", "AC_Hours_Per_Day", "AC_Usage_Band"]].head(10))
print("\nUsage_Status counts:\n", df["Usage_Status"].value_counts())
print("\nAC_Usage_Band counts:\n", df["AC_Usage_Band"].value_counts())
print("\nMissing values in new columns:\n",
      df[["Usage_Per_Sqft", "Usage_Status", "AC_Usage_Band"]].isna().sum())


# ------------------------------------------------------------------
# 6. SAVE (the cleaned CSV in data/processed now includes the new columns)
# ------------------------------------------------------------------
df.to_csv(CLEAN_PATH, index=False)
print("\nSaved:", CLEAN_PATH, df.shape)


Loaded: (250, 12)
  Household_ID  Monthly_Consumption_kWh  Home_Area_Sqft  Usage_Per_Sqft Usage_Status  AC_Hours_Per_Day AC_Usage_Band
0        H1091                      166             900           0.184       Normal               1.5     Low (0-3)
1        H1133                      324            1550           0.209       Normal               7.9    Heavy (7+)
2        H1084                      402            3020           0.133   High Usage               7.1    Heavy (7+)
3        H1056                      175             740           0.236       Normal               5.3  Medium (3-7)
4        H1129                      217             460           0.472       Normal               7.4    Heavy (7+)
5        H1061                      262             990           0.265       Normal               5.5  Medium (3-7)
6        H1102                      167            1190           0.140       Normal               6.9  Medium (3-7)
7        H1070                      309       

Here we will generate the visual images of prossed data.


In [16]:
import os
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

# Run from the "notebooks" folder, after clean_data.py and create_new_columns.py.
CLEAN_PATH = "../data/processed/electricity_data_clean.csv"
IMG_DIR = "../images"
OUTCOME = "Monthly_Consumption_kWh"
THRESHOLD = 350            # High Usage threshold (same as the SRS)

os.makedirs(IMG_DIR, exist_ok=True)
sns.set_theme(style="whitegrid", rc={"axes.grid.axis": "y"})
pd.set_option("display.width", 150)
pd.set_option("display.max_columns", 20)


def save_chart(fig, filename):
    """Save a figure as a PNG in the images folder and close it."""
    fig.tight_layout()
    fig.savefig(os.path.join(IMG_DIR, filename), dpi=150)
    plt.close(fig)
    print("Saved chart:", filename)


# ------------------------------------------------------------------
# 0. LOAD THE CLEANED DATA (with the three new columns)
# ------------------------------------------------------------------
df = pd.read_csv(CLEAN_PATH, parse_dates=["Reading_Date"])
band_order = ["Low (0-3)", "Medium (3-7)", "Heavy (7+)"]
df["AC_Usage_Band"] = pd.Categorical(df["AC_Usage_Band"], categories=band_order, ordered=True)
print("Loaded:", df.shape)


# ==================================================================
# QUESTION 1: What does Monthly_Consumption_kWh look like overall?
# Method: summary statistics.  Chart: histogram (+ box plot)
# ==================================================================
print("\n" + "=" * 70 + "\nQUESTION 1: Overall consumption\n" + "=" * 70)
q1_table = df[OUTCOME].describe().round(1)
print(q1_table)
print("Median:", df[OUTCOME].median(), "| Skewness:", round(df[OUTCOME].skew(), 2))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.5), gridspec_kw={"width_ratios": [3, 1]})
ax1.hist(df[OUTCOME], bins=20, color="#4C78A8", edgecolor="white")
ax1.axvline(df[OUTCOME].mean(), color="red", linestyle="--", label=f"Mean = {df[OUTCOME].mean():.0f} kWh")
ax1.axvline(df[OUTCOME].median(), color="green", linestyle="-", label=f"Median = {df[OUTCOME].median():.0f} kWh")
ax1.axvline(THRESHOLD, color="black", linestyle=":", label=f"High Usage threshold = {THRESHOLD} kWh")
ax1.set_title("Q1: Distribution of monthly electricity consumption")
ax1.set_xlabel("Monthly consumption (kWh)")
ax1.set_ylabel("Number of households")
ax1.legend()
sns.boxplot(y=df[OUTCOME], ax=ax2, color="#4C78A8")
ax2.set_title("Box plot")
ax2.set_ylabel("Monthly consumption (kWh)")
save_chart(fig, "q1_consumption_histogram.png")


# ==================================================================
# QUESTION 2: Do AC_Hours_Per_Day and Home_Area_Sqft affect the outcome?
# Method: correlation + average usage by AC_Usage_Band.
# Charts: scatter plots, correlation heatmap, bar chart by AC band
# ==================================================================
print("\n" + "=" * 70 + "\nQUESTION 2: Effect of AC hours and home area\n" + "=" * 70)
num_cols = ["Household_Size", "Home_Area_Sqft", "AC_Hours_Per_Day",
            "Avg_Temperature_C", "Appliance_Age_Years", OUTCOME]
corr = df[num_cols].corr().round(2)
q2_corr = corr[OUTCOME].drop(OUTCOME).sort_values(ascending=False)
print("Correlation of each numeric column with", OUTCOME, ":\n", q2_corr)

q2_band = (df.groupby("AC_Usage_Band", observed=True)[OUTCOME]
             .agg(Households="count", Average_kWh="mean", Median_kWh="median").round(1))
print("\nAverage consumption by AC usage band:\n", q2_band)

# Chart 2a: scatter plots with a trend line
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, col, color in zip(axes, ["AC_Hours_Per_Day", "Home_Area_Sqft"], ["#E45756", "#54A24B"]):
    ax.scatter(df[col], df[OUTCOME], alpha=0.6, color=color, edgecolor="white")
    slope, intercept = np.polyfit(df[col], df[OUTCOME], 1)
    xs = np.linspace(df[col].min(), df[col].max(), 100)
    ax.plot(xs, slope * xs + intercept, color="black", linewidth=2)
    r = df[col].corr(df[OUTCOME])
    ax.set_title(f"{col} vs consumption (r = {r:.2f})")
    ax.set_xlabel(col)
    ax.set_ylabel("Monthly consumption (kWh)")
save_chart(fig, "q2_scatter_ac_hours_and_area.png")

# Chart 2b: correlation heatmap
fig, ax = plt.subplots(figsize=(7.5, 6))
sns.heatmap(corr, annot=True, cmap="coolwarm", vmin=-1, vmax=1, fmt=".2f", ax=ax)
ax.set_title("Q2: Correlation between numeric columns")
save_chart(fig, "q2_correlation_heatmap.png")

# Chart 2c: average consumption by AC usage band
fig, ax = plt.subplots(figsize=(6.5, 4.5))
bars = ax.bar(q2_band.index.astype(str), q2_band["Average_kWh"], color=["#9ecae9", "#4C78A8", "#1f4e79"])
ax.bar_label(bars, fmt="%.0f")
ax.set_title("Q2: Average consumption by AC usage band")
ax.set_xlabel("AC usage band (hours per day)")
ax.set_ylabel("Average monthly consumption (kWh)")
save_chart(fig, "q2_usage_by_ac_band.png")


# ==================================================================
# QUESTION 3: Does the outcome differ by City, Property_Type,
#             Tariff_Plan and Has_Solar?
# Method: group averages.  Chart: bar charts of average usage
# ==================================================================
print("\n" + "=" * 70 + "\nQUESTION 3: Differences between groups\n" + "=" * 70)
cat_cols = ["City", "Property_Type", "Tariff_Plan", "Has_Solar"]
q3_tables = {}
for col in cat_cols:
    t = (df.groupby(col)[OUTCOME].agg(Households="count", Average_kWh="mean", Median_kWh="median")
           .round(1).sort_values("Average_kWh", ascending=False))
    q3_tables[col] = t
    print(f"\nAverage consumption by {col}:\n", t)

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
colors = ["#4C78A8", "#F58518", "#54A24B", "#B279A2"]
for ax, col, color in zip(axes.ravel(), cat_cols, colors):
    t = q3_tables[col]
    bars = ax.bar(t.index, t["Average_kWh"], color=color)
    ax.bar_label(bars, fmt="%.0f")
    ax.set_title(f"Average consumption by {col}")
    ax.set_xlabel(col)
    ax.set_ylabel("Average kWh per month")
    ax.set_ylim(0, t["Average_kWh"].max() * 1.15)
save_chart(fig, "q3_usage_by_category.png")


# ==================================================================
# QUESTION 4: Which households need attention?
# High Usage = Monthly_Consumption_kWh above 350
# Method: count and share of High Usage + comparison with Normal.
# Charts: count plot of Usage_Status, High Usage share by City
# ==================================================================
print("\n" + "=" * 70 + "\nQUESTION 4: Households that need attention\n" + "=" * 70)
q4_counts = df["Usage_Status"].value_counts()
q4_share = (df["Usage_Status"].value_counts(normalize=True) * 100).round(1)
print("Number of households:\n", q4_counts)
print("\nShare of households (%):\n", q4_share)

compare_cols = ["Home_Area_Sqft", "AC_Hours_Per_Day", "Household_Size",
                "Avg_Temperature_C", "Appliance_Age_Years", OUTCOME]
q4_compare = df.groupby("Usage_Status")[compare_cols].mean().round(1).T
print("\nAverage of each column, High Usage vs Normal:\n", q4_compare)

for col in ["City", "Property_Type", "Has_Solar"]:
    share = (df.groupby(col)["Usage_Status"].apply(lambda s: (s == "High Usage").mean() * 100)
               .round(1).sort_values(ascending=False))
    print(f"\nHigh Usage share (%) by {col}:\n", share)

# Chart 4a: count plot of Usage_Status
fig, ax = plt.subplots(figsize=(5.5, 4.5))
sns.countplot(data=df, x="Usage_Status", order=["Normal", "High Usage"],
              palette=["#4C78A8", "#E45756"], hue="Usage_Status", legend=False, ax=ax)
ax.bar_label(ax.containers[0])
ax.bar_label(ax.containers[1])
ax.set_title(f"Q4: Households by usage status (High Usage > {THRESHOLD} kWh)")
ax.set_xlabel("Usage status")
ax.set_ylabel("Number of households")
save_chart(fig, "q4_usage_status_count.png")

# Chart 4b: share of High Usage households by city
share_city = (df.groupby("City")["Usage_Status"].apply(lambda s: (s == "High Usage").mean() * 100)
                .sort_values(ascending=False))
fig, ax = plt.subplots(figsize=(7, 4.5))
bars = ax.bar(share_city.index, share_city.values, color="#E45756")
ax.bar_label(bars, fmt="%.1f%%")
ax.set_title("Q4: Share of High Usage households by city")
ax.set_xlabel("City")
ax.set_ylabel("High Usage households (%)")
ax.set_ylim(0, share_city.max() * 1.2)
save_chart(fig, "q4_high_usage_share_by_city.png")

print("\nDone. Charts are saved in:", os.path.abspath(IMG_DIR))


Loaded: (250, 15)

QUESTION 1: Overall consumption
count    250.0
mean     245.8
std       82.4
min       58.0
25%      190.2
50%      240.0
75%      300.2
max      529.0
Name: Monthly_Consumption_kWh, dtype: float64
Median: 240.0 | Skewness: 0.28
Saved chart: q1_consumption_histogram.png

QUESTION 2: Effect of AC hours and home area
Correlation of each numeric column with Monthly_Consumption_kWh :
 AC_Hours_Per_Day       0.59
Home_Area_Sqft         0.50
Avg_Temperature_C      0.37
Household_Size        -0.01
Appliance_Age_Years   -0.09
Name: Monthly_Consumption_kWh, dtype: float64

Average consumption by AC usage band:
                Households  Average_kWh  Median_kWh
AC_Usage_Band                                     
Low (0-3)              70        187.1       179.5
Medium (3-7)          120        248.3       243.0
Heavy (7+)             60        309.2       300.0
Saved chart: q2_scatter_ac_hours_and_area.png
Saved chart: q2_correlation_heatmap.png
Saved chart: q2_usage_by_ac_ba